# Run an Experiment

Config-first workflow:

1. Pick an experiment id from `configs/experiments/`.
2. Call `run_experiment(cfg)`. That's it.

To add a new experiment, drop a YAML into `configs/experiments/` — no code change.
Every run snapshots its fully-resolved config into `data/processed/xgboost_experiments/<id>/config_snapshot.json` so results are reproducible.

In [20]:
# Make the src/ package importable when running from notebooks/
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import list_experiments, load_config
from src.experiments.runner import run_experiment
from src.viz import make_all_plots

print('Available experiments:')
for name in list_experiments():
    print(' -', name)

In [26]:
# Pick one and load it. Change this single line to switch experiments.
ACTIVE = 'exp_010_loo_only'
cfg = load_config(ACTIVE)

print(f'Loaded: {cfg.id}')
print(f'  description : {cfg.description}')
print(f'  hypothesis  : {cfg.hypothesis}')
print(f'  sections    : {cfg.run.sections}')
print(f'  trials      : {cfg.xgb.n_trials}')
print(f'  engineered  : {cfg.features.use_engineered}')
print(f'  results dir : {cfg.results_dir}')

Loaded: exp_010_loo_only
  description : Leave-one-out generalisability evaluation (row + mix variants).
  hypothesis  : LOO reveals strict generalisability; scores drop vs train/test split.
  sections    : ['loo']
  trials      : 10
  engineered  : True
  results dir : /Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_010_loo_only


## Run

`run_experiment` loads splits, fits physics parameters for every model, and dispatches to each section listed in `cfg.run.sections`. Each run gets a unique directory under `data/processed/xgboost_experiments/` named `{experiment_id}_{YYYYMMDD_HHMMSS}`, so re-running the same config never overwrites previous results. The path is available as `results.results_dir`.

In [27]:
results = run_experiment(cfg)

ACTIVE EXPERIMENT: exp_010_loo_only
Description : Leave-one-out generalisability evaluation (row + mix variants).
Hypothesis  : LOO reveals strict generalisability; scores drop vs train/test split.
Sections    : ['loo']
Trials      : 10
Engineered  : True
Log numeric : False
Seed        : 42

Loaded splits: train=136 rows, test=42 rows
Built physics datasets for 5 models.

LOO variant: exp_loo_row (leave_out_unit=row)
  5-parameter CY     | mean R²=0.3002
  Power Law          | mean R²=0.6472
  Sisko              | mean R²=0.3261
  Williamson         | mean R²=0.2475
  Constrained CY     | mean R²=0.4475

LOO variant: exp_loo_mix (leave_out_unit=mix)
  5-parameter CY     | mean R²=0.2415
  Power Law          | mean R²=0.6443
  Sisko              | mean R²=0.1797
  Williamson         | mean R²=0.1193
  Constrained CY     | mean R²=0.4237

Results written to: /Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/

## Inspect results

`results` is an `ExperimentResults`: a dict of `SectionResult` objects keyed by section name. Every section has `model_summary`, `per_target`, and (optionally) `predictions` DataFrames.

In [28]:
for name, section in results.sections.items():
    print(f'--- {name} ---')
    display(section.sorted_by_r2().head())

print('Best (baseline/sweep/loo, first hit):', results.best())

--- loo ---


,variant_id,leave_out_unit,model_name,model_key,test_r2,test_mse,test_rmse
0,exp_loo_row,row,Power Law,power_law,0.647151,1310.779660,36.204691
1,exp_loo_mix,mix,Power Law,power_law,0.644326,1321.272546,36.349313
2,exp_loo_row,row,Constrained CY,constrained_cy,0.447507,40814.397348,202.025734
3,exp_loo_mix,mix,Constrained CY,constrained_cy,0.423694,42573.512468,206.333498
4,exp_loo_row,row,Sisko,sisko,0.326114,0.127222,0.356682


Best (baseline/sweep/loo, first hit): None


## Plots

In [29]:
saved = make_all_plots(results, results.results_dir)
for p in saved:
    print(p)

## Compare completed experiments

The `experiment_manifest.json` at the top of `data/processed/xgboost_experiments/` accumulates every run across the project.

In [30]:
import json
import pandas as pd

manifest_path = cfg.experiments_root_path / 'experiment_manifest.json'
if manifest_path.exists():
    runs = json.loads(manifest_path.read_text())['runs']
    df = pd.json_normalize(runs)
    cols = ['run_id', 'experiment_id', 'description', 'sections_run',
            'best.model_name', 'best.test_r2', 'duration_seconds', 'created_at_utc']
    display(df[[c for c in cols if c in df.columns]])
else:
    print('No manifest yet. Run an experiment first.')